In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src import fdm_schemes
import matplotlib.animation as animation


In [ ]:

L = 1.0
N = 1001
dx = L / (N-1)
dt = 0.001
T = 10.0
c = 1.0


u0 = np.sin(2 * np.pi * np.linspace(0, L, N))

# u0[:np.ceil(1/5*(N-1))] = 0
# u0[np.floor(2/5*(N-1))+1:] = 0

res = fdm_schemes.wave_equation_1d(u0, c, dx, dt, T)

In [ ]:
plt.imshow(res, aspect='auto', extent=(0, L, T, 0))

In [ ]:
plt.plot(res.T[:,0::100])

In [ ]:
# Animation helper: create and save an animation of the 1D wave
# The user can set `out_filename` below before running this cell.

def save_wave_animation(u_time_space, x, dt, out_filename='wave_animation.gif', frame_step=None, dpi=150):
    """
    Create and save an animation of a 1D wave solution.

    Parameters
    ----------
    u_time_space : ndarray
        Array of shape (nt, nx) containing solution at each time step.
    x : ndarray
        Spatial coordinates of length nx.
    dt : float
        Time step used between frames (seconds).
    out_filename : str
        Output filename (recommended extension: .gif or .mp4).
    frame_step : int or None
        Save every `frame_step`-th frame. If None, it will be chosen so the
        output has at most ~200 frames.
    dpi : int
        Resolution for the saved animation.
    """
    nt, nx = u_time_space.shape

    if frame_step is None:
        frame_step = max(1, int(nt / 200))

    frames = list(range(0, nt, frame_step))
    interval_ms = dt * 1000 * frame_step

    fig, ax = plt.subplots()
    line, = ax.plot(x, u_time_space[0])
    ax.set_xlim(x.min(), x.max())
    y_margin = 0.1 * (u_time_space.max() - u_time_space.min())
    if y_margin == 0:
        y_margin = 1.0
    ax.set_ylim(u_time_space.min() - y_margin, u_time_space.max() + y_margin)
    ax.set_xlabel('x')
    ax.set_ylabel('u')
    ax.set_title('1D wave')

    def update(i):
        line.set_ydata(u_time_space[i])
        ax.set_title(f'Time = {i*dt:.3f} s')
        return (line,)

    anim = animation.FuncAnimation(fig, update, frames=frames, interval=interval_ms, blit=True)

    # Try to use PillowWriter (no external ffmpeg required) when saving GIFs
    try:
        from matplotlib.animation import PillowWriter
        if out_filename.lower().endswith('.gif'):
            writer = PillowWriter(fps=max(1, int(1000/interval_ms)))
            anim.save(out_filename, writer=writer, dpi=dpi)
        else:
            # For mp4, fall back to ffmpeg if available
            anim.save(out_filename, dpi=dpi)
    except Exception:
        # Fallback: try default save (may require ffmpeg)
        anim.save(out_filename, dpi=dpi)

    plt.close(fig)
    print(f"Saved animation to: {out_filename}")


# User-configurable output filename (change this before running)
out_filename = 'assignment1_2pi.gif'
# Build x vector and call the saver
x = np.linspace(0, L, N)
save_wave_animation(res, x, dt, out_filename=out_filename)
